<a href="https://colab.research.google.com/github/pmetheny99/hello-world/blob/main/PNC_1_20_titoli_ultimo_anno_ver2_5-8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# =============================================================================
# NOME SCRIPT: nexi_pnc_analisi.py
# VERSIONE: 2.5.8 (Revisione del 17-04-2024)
# DESCRIZIONE: Analisi YTD PNC con calcolo Ricopertura e Excel Professionale.
# REVISIONI:
# - v2.5.7: Styling avanzato Excel (allineamenti, colori status, bordi).
# - v2.5.8: Centratura data e gestione "n.d." per ricopertura zero.
# =============================================================================

import os
import subprocess
import sys
import time
import warnings

# Silenzia i warning di openpyxl relativi agli stili mancanti nel file Consob
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# --- PARAMETRI DI CONFIGURAZIONE ---
NUMERO_TITOLI_DA_ANALIZZARE = 10
# Se vuoi analizzare un solo titolo specifico, inserisci il nome esatto qui sotto (es: "NEXI SPA")
TITOLO_SPECIFICO = ""
# -----------------------------------

def setup_environment():
    """Installa le librerie necessarie se mancano."""
    packages = ['openpyxl', 'plotly', 'pandas']
    for p in packages:
        try:
            __import__(p)
        except ImportError:
            print(f"📦 Installazione libreria: {p}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", p])

    try:
        import kaleido
    except ImportError:
        print("📦 Installazione motore grafico (Kaleido)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "kaleido==0.1.0.post1"])

setup_environment()

import pandas as pd
import plotly.graph_objects as go
import zipfile
from datetime import datetime
from google.colab import files
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# Codici colore ANSI per terminale
class Colori:
    VERDE = '\033[92m'    # Forte Ricopertura
    GIALLO = '\033[93m'   # In Ricopertura
    ROSSO = '\033[91m'    # Sui Massimi
    GRIGIO = '\033[90m'   # Neutro
    RESET = '\033[0m'
    BOLD = '\033[1m'

def analizza_pnc_ytd(n_top_titoli=5, titolo_target=""):
    print(f"📊 Avvio analisi PNC YTD v2.5.8 (Update: {datetime.now().strftime('%d/%m/%Y')})")

    # 1. Rilevamento file
    all_files = os.listdir()
    excel_files = [f for f in all_files if 'PncPubbl' in f and f.endswith('.xlsx')]

    if not excel_files:
        print("\n❌ ERRORE: Caricare il file 'PncPubbl.xlsx' su Colab prima di avviare!")
        return

    f_excel = excel_files[0]

    # 2. Caricamento Dati
    try:
        xl = pd.ExcelFile(f_excel)
        sheet_curr = [s for s in xl.sheet_names if 'Correnti' in s or 'Current' in s][0]
        sheet_hist = [s for s in xl.sheet_names if 'Storiche' in s or 'Historic' in s][0]
        df_curr = pd.read_excel(f_excel, sheet_name=sheet_curr)
        df_hist = pd.read_excel(f_excel, sheet_name=sheet_hist)
    except Exception as e:
        print(f"❌ Errore caricamento Excel: {e}")
        return

    col_perc = 'Perc. posizione netta corta'
    col_data = 'Data della posizione'
    col_emittente = 'Emittente'
    col_detentore = 'Detentore'

    def clean_df(df):
        df[col_perc] = pd.to_numeric(df[col_perc], errors='coerce')
        df[col_data] = pd.to_datetime(df[col_data], dayfirst=True, errors='coerce', format='mixed')
        return df.dropna(subset=[col_perc, col_data])

    df_curr, df_hist = clean_df(df_curr), clean_df(df_hist)
    df_all = pd.concat([df_curr, df_hist], ignore_index=True)

    # 3. Selezione Titoli
    if titolo_target and titolo_target.strip() != "":
        if titolo_target in df_all[col_emittente].unique():
            print(f"🎯 Focus prioritario su Azione: {titolo_target}")
            top_titoli = [titolo_target]
        else:
            print(f"⚠️ Azione '{titolo_target}' non trovata. Procedo con la Top {n_top_titoli}.")
            top_titoli = df_curr.groupby(col_emittente)[col_perc].sum().sort_values(ascending=False).head(n_top_titoli).index.tolist()
    else:
        top_titoli = df_curr.groupby(col_emittente)[col_perc].sum().sort_values(ascending=False).head(n_top_titoli).index.tolist()

    # 4. Timeline YTD
    start_ytd = pd.Timestamp(year=datetime.now().year, month=1, day=1)
    end_ytd = pd.Timestamp.now().normalize()
    date_range = pd.date_range(start=start_ytd, end=end_ytd, freq='B')

    fig_main = go.Figure()

    colors_plot = [
        '#32CD32', '#FF4500', '#1E90FF', '#FFD700', '#FF00FF', '#00FFFF', '#FFA500', '#ADFF2F',
        '#FF69B4', '#8A2BE2', '#00FF7F', '#F4A460', '#D2691E', '#7FFF00', '#DC143C', '#00CED1'
    ]

    all_data_log = []
    summary_list = []

    for i, emittente in enumerate(top_titoli):
        df_t = df_all[df_all[col_emittente] == emittente].copy()
        color_p = colors_plot[i % len(colors_plot)]
        var_dates = df_t[col_data].dt.normalize().unique()

        daily_pnc = []
        for d in date_range:
            snap = df_t[df_t[col_data] <= d]
            if snap.empty:
                daily_pnc.append({'Data': d, 'PNC': 0.0, 'Var': False})
                continue
            val = snap.sort_values(col_data).groupby(col_detentore).tail(1)
            total = val[val[col_perc] >= 0.5][col_perc].sum()
            daily_pnc.append({'Data': d, 'PNC': total, 'Var': d in var_dates})

        df_d = pd.DataFrame(daily_pnc)

        # Calcolo Metriche Ricopertura
        curr_val = df_d['PNC'].iloc[-1]
        pnc_max = df_d['PNC'].max()

        perc_ricop = 0.0
        if pnc_max > 0:
            perc_ricop = ((pnc_max - curr_val) / pnc_max) * 100

        # Gestione visualizzazione Ricopertura (n.d. se 0.0)
        ricop_display = round(perc_ricop, 1) if perc_ricop > 0 else "n.d."

        # Definizione Status e Colore
        if perc_ricop > 20:
            status = "Forte Ricopertura"
            col_ansi = Colori.VERDE
        elif perc_ricop > 5:
            status = "In Ricopertura"
            col_ansi = Colori.GIALLO
        else:
            status = "Sui Massimi"
            col_ansi = Colori.ROSSO

        summary_list.append({
            'Azione': emittente,
            'PNC Att. %': round(curr_val, 2),
            'Data': df_d['Data'].iloc[-1].strftime('%d/%m'),
            'PNC Max %': round(pnc_max, 2),
            '% Ricop.': ricop_display,
            'Status': status,
            '_colore': col_ansi
        })

        # Plot Principale
        fig_main.add_trace(go.Scatter(
            x=df_d['Data'], y=df_d['PNC'], name=emittente, mode='lines+markers',
            marker=dict(size=[7 if v else 0 for v in df_d['Var']], color=color_p),
            line=dict(color=color_p, width=2),
            customdata=[[pnc_max, perc_ricop]]*len(df_d),
            text=[emittente]*len(df_d),
            hovertemplate="<b>%{text}</b><br>Data: %{x|%d %b}<br>PNC: %{y:.2f}%<br>Max YTD: %{customdata[0]:.2f}%<br>Ricopertura: %{customdata[1]:.1f}%<extra></extra>"
        ))

        df_d['Azione'] = emittente
        df_d['PNC_Max_YTD'] = pnc_max
        df_d['Perc_Ricopertura'] = perc_ricop
        all_data_log.append(df_d)

    # Output a schermo
    L_AZIONE = 45
    TAB_WIDTH = 105
    print("\n" + "=" * TAB_WIDTH)
    header = f"{'Azione':<{L_AZIONE}} | {'PNC %':<7} | {'Data':<5} | {'Max %':<7} | {'% Ricop':<8} | {'Status'}"
    print(Colori.BOLD + header + Colori.RESET)
    print("-" * TAB_WIDTH)

    for row in summary_list:
        nome_display = (row['Azione'][:L_AZIONE-3] + '..') if len(row['Azione']) > L_AZIONE else row['Azione']
        ricop_str = f"{row['% Ricop.']}%" if row['% Ricop.'] != "n.d." else "n.d.   "
        linea = f"{nome_display:<{L_AZIONE}} | {row['PNC Att. %']:<7.2f} | {row['Data']:<5} | {row['PNC Max %']:<7.2f} | {ricop_str:<8} | {row['Status']}"
        print(row['_colore'] + linea + Colori.RESET)
    print("=" * TAB_WIDTH)

    fig_main.update_layout(
        template="plotly_dark", height=600, width=1100,
        title="Evoluzione PNC Totali (YTD) - Analisi Ricoperture",
        xaxis=dict(rangebreaks=[dict(bounds=["sat", "mon"])], tickformat="%d %b", title="Data"),
        yaxis=dict(title="PNC Totale %"),
        legend=dict(font=dict(size=9))
    )
    fig_main.show()

    # ESPORTAZIONE EXCEL PROFESSIONALE
    xlsx_file = f"Report_PNC_v2.5.8.xlsx"
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        # 1. Foglio Dettagliato
        pd.concat(all_data_log).to_excel(writer, sheet_name='Storico_Dettagliato', index=False)

        # 2. Foglio Sintesi
        df_sintesi = pd.DataFrame(summary_list).drop(columns=['_colore'])
        df_sintesi.to_excel(writer, sheet_name='Sintesi', index=False)

        # Applicazione Styling
        workbook = writer.book
        ws = workbook['Sintesi']

        # Colori e Font
        header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
        header_font = Font(color='FFFFFF', bold=True)
        center_ali = Alignment(horizontal='center', vertical='center')
        left_ali = Alignment(horizontal='left', vertical='center')
        thin_border = Border(left=Side(style='thin'), right=Side(style='thin'),
                             top=Side(style='thin'), bottom=Side(style='thin'))

        # Fills per Status
        fill_green = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid') # Forte
        fill_yellow = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid') # In Ric
        fill_red = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid') # Sui Max

        # Styling Intestazioni
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = center_ali
            cell.border = thin_border

        # Styling Righe
        for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
            status_val = ws.cell(row=row_idx, column=6).value # Colonna Status

            for cell in row:
                cell.border = thin_border

                # Allineamento dinamico
                if cell.column == 1:
                    cell.alignment = left_ali   # Azione
                else:
                    cell.alignment = center_ali # Tutti gli altri centrati (inclusa Data)

                # Formattazione numerica se non è "n.d."
                if cell.column in [2, 4, 5]:
                    if cell.value != "n.d.":
                        cell.number_format = '0.00'

                # Colore condizionale in base allo Status
                if status_val == "Forte Ricopertura":
                    ws.cell(row=row_idx, column=6).fill = fill_green
                elif status_val == "In Ricopertura":
                    ws.cell(row=row_idx, column=6).fill = fill_yellow
                elif status_val == "Sui Massimi":
                    ws.cell(row=row_idx, column=6).fill = fill_red

        # Adattamento larghezza colonne
        dims = {}
        for row in ws.rows:
            for cell in row:
                if cell.value:
                    dims[cell.column_letter] = max((dims.get(cell.column_letter, 0), len(str(cell.value))))
        for col, value in dims.items():
            ws.column_dimensions[col].width = value + 5

    # Salvataggio HTML e ZIP
    fig_main.write_html("Analisi_PNC.html")
    zip_fn = "Analisi_PNC_v2.5.8_Updated.zip"
    with zipfile.ZipFile(zip_fn, 'w') as z:
        z.write("Analisi_PNC.html")
        z.write(xlsx_file)

    print(f"\n✅ ZIP generato con Excel Aggiornato (Data centrata e 'n.d.'): {zip_fn}")
    files.download(zip_fn)

# Esecuzione
analizza_pnc_ytd(n_top_titoli=NUMERO_TITOLI_DA_ANALIZZARE, titolo_target=TITOLO_SPECIFICO)

📊 Avvio analisi PNC YTD v2.5.8 (Update: 17/04/2026)

Azione                                        | PNC %   | Data  | Max %   | % Ricop  | Status
---------------------------------------------------------------------------------------------------------
SAIPEM SPA                                    | 10.69   | 17/04 | 13.14   | 18.6%    | In Ricopertura
BFF BANK SPA                                  | 6.25    | 17/04 | 6.25    | n.d.     | Sui Massimi
NEXI SPA                                      | 5.91    | 17/04 | 7.89    | 25.1%    | Forte Ricopertura
AVIO SPA                                      | 4.25    | 17/04 | 4.61    | 7.8%     | In Ricopertura
DIASORIN SPA                                  | 3.43    | 17/04 | 3.43    | n.d.     | Sui Massimi
BRUNELLO CUCINELLI SPA                        | 3.26    | 17/04 | 4.08    | 20.1%    | Forte Ricopertura
STELLANTIS NV                                 | 3.12    | 17/04 | 3.15    | 1.0%     | Sui Massimi
AMPLIFON SPA                        


✅ ZIP generato con Excel Aggiornato (Data centrata e 'n.d.'): Analisi_PNC_v2.5.8_Updated.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>